In [ ]:
import pandas as pd

print("1. Loading Master Coordinates...")
# Load the verified coordinates generated from Notebook 1
master_cameras = pd.read_csv('../data/nyc_camera_coordinates.csv')
valid_intersections = set(master_cameras['Clean_Intersection'])

# Assuming fy2017_df through fy2027_df are already loaded in memory or via DuckDB
yearly_dataframes = [
    fy2017_df, fy2018_df, fy2019_df, fy2020_df, 
    fy2021_df, fy2022_df, fy2023_df, fy2024_df, 
    fy2025_df, fy2026_df, fy2027_df
]

all_historical_counts = []

print("2. Processing Historical Timelines...")
for year_idx, df in enumerate(yearly_dataframes, start=2017):
    print(f"Processing FY{year_idx}...")
    
    # Standardize column names in case API versions differ across years
    df.columns = df.columns.str.lower()
    
    # Ensure necessary columns exist before processing
    if not all(col in df.columns for col in ['street_name', 'intersecting_street', 'issue_date']):
        continue

    # 1. Clean the text using the exact same logic as the spatial map
    raw_text = df['street_name'].fillna('') + df['intersecting_street'].fillna('')
    df['Clean_Intersection'] = (
        raw_text
        .str.replace(r'\b(NB|SB|EB|WB|N/B|S/B|E/B|W/B)\b', '', regex=True)
        .str.replace(r'\(.*?\)', '', regex=True)
        .str.replace('@', ' AND ')
        .str.replace(r'^\s+', '', regex=True)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
        + ', NYC'
    )
    
    # 2. Filter out random text strings that aren't verified camera locations
    df = df[df['Clean_Intersection'].isin(valid_intersections)].copy()
    
    # 3. Extract the Year and Month from the issue_date
    df['issue_date'] = pd.to_datetime(df['issue_date'], errors='coerce')
    df['YearMonth'] = df['issue_date'].dt.to_period('M')
    
    # 4. Group by Intersection and Month to get the time-series curve
    monthly_counts = df.groupby(['Clean_Intersection', 'YearMonth']).size().reset_index(name='Monthly_Tickets')
    all_historical_counts.append(monthly_counts)

print("\n3. Compiling Final Time-Series Dataset...")
# Combine all 11 years into a single timeline
historical_timeline = pd.concat(all_historical_counts, ignore_index=True)

# Sum up any overlapping months across datasets
historical_timeline = historical_timeline.groupby(['Clean_Intersection', 'YearMonth'])['Monthly_Tickets'].sum().reset_index()

# Sort chronologically for changepoint detection
historical_timeline = historical_timeline.sort_values(by=['Clean_Intersection', 'YearMonth'])

output_path = '../data/camera_historical_timeline.csv'
historical_timeline.to_csv(output_path, index=False)
print(f"Saved 11-year historical timeline to: {output_path}")
print(historical_timeline.head(10))